# Comprehensive Network Security Analyzer

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
# Cybersecurity Network Security Analyzer
# A comprehensive tool for network analysis, vulnerability detection, and security monitoring

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import hashlib
import base64
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

class NetworkSecurityAnalyzer:
    def __init__(self):
        self.suspicious_ips = set()
        self.blocked_domains = set()
        self.attack_patterns = {
            'sql_injection': [r"union\s+select", r"drop\s+table", r"insert\s+into", r"'.*or.*'.*=.*'"],
            'xss': [r"<script.*>", r"javascript:", r"onerror\s*=", r"onload\s*="],
            'directory_traversal': [r"\.\.\/", r"\.\.\\", r"%2e%2e%2f", r"%2e%2e%5c"],
            'command_injection': [r";\s*cat\s+", r";\s*ls\s+", r"&&\s*", r"\|\s*"]
        }

    def generate_sample_logs(self, num_logs=1000):
        """Generate sample network logs for analysis"""
        log_types = ['GET', 'POST', 'PUT', 'DELETE']
        status_codes = [200, 301, 302, 400, 401, 403, 404, 500, 502, 503]

        # Benign IPs
        benign_ips = [f"192.168.1.{i}" for i in range(1, 50)]
        benign_ips.extend([f"10.0.0.{i}" for i in range(1, 30)])

        # Suspicious IPs (simulate attackers)
        suspicious_ips = [f"185.220.{random.randint(1, 255)}.{random.randint(1, 255)}" for _ in range(10)]
        suspicious_ips.extend([f"142.{random.randint(1, 255)}.{random.randint(1, 255)}.{random.randint(1, 255)}" for _ in range(5)])

        logs = []
        base_time = datetime.now() - timedelta(days=7)

        for i in range(num_logs):
            # 90% benign traffic, 10% suspicious
            if random.random() < 0.9:
                ip = random.choice(benign_ips)
                url = random.choice(['/home', '/login', '/dashboard', '/api/users', '/static/css/style.css'])
                method = random.choice(['GET', 'GET', 'GET', 'POST'])  # More GETs
                status = random.choice([200, 200, 200, 301, 404])  # Mostly successful
            else:
                ip = random.choice(suspicious_ips)
                # Inject attack patterns
                attack_urls = [
                    "/admin' OR '1'='1",
                    "/search?q=<script>alert('xss')</script>",
                    "/file?path=../../etc/passwd",
                    "/cmd?exec=cat /etc/shadow",
                    "/login.php?user=admin'--",
                    "/upload.php?file=../../../config.php"
                ]
                url = random.choice(attack_urls)
                method = random.choice(['POST', 'GET'])
                status = random.choice([400, 401, 403, 500])

            timestamp = base_time + timedelta(minutes=random.randint(0, 10080))  # Within 7 days
            user_agent = random.choice([
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
                'curl/7.68.0',
                'sqlmap/1.4.7',
                'Mozilla/5.0 (compatible; scanner/1.0)',
                'python-requests/2.25.1'
            ])

            bytes_sent = random.randint(100, 50000)

            logs.append({
                'timestamp': timestamp,
                'ip_address': ip,
                'method': method,
                'url': url,
                'status_code': status,
                'bytes': bytes_sent,
                'user_agent': user_agent
            })

        return pd.DataFrame(logs)

    def detect_attacks(self, logs_df):
        """Detect various attack patterns in logs"""
        attacks_detected = []

        for idx, log in logs_df.iterrows():
            url = log['url'].lower()
            user_agent = log['user_agent'].lower()

            for attack_type, patterns in self.attack_patterns.items():
                for pattern in patterns:
                    if re.search(pattern, url, re.IGNORECASE) or re.search(pattern, user_agent, re.IGNORECASE):
                        attacks_detected.append({
                            'timestamp': log['timestamp'],
                            'ip_address': log['ip_address'],
                            'attack_type': attack_type,
                            'url': log['url'],
                            'severity': self._calculate_severity(attack_type, log['status_code'])
                        })
                        self.suspicious_ips.add(log['ip_address'])
                        break

        return pd.DataFrame(attacks_detected)

    def _calculate_severity(self, attack_type, status_code):
        """Calculate attack severity based on type and response"""
        base_severity = {
            'sql_injection': 8,
            'xss': 6,
            'directory_traversal': 7,
            'command_injection': 9
        }

        severity = base_severity.get(attack_type, 5)

        # Successful attacks (200 status) are more severe
        if status_code == 200:
            severity = min(10, severity + 2)
        elif status_code in [403, 401]:
            severity = max(1, severity - 1)  # Blocked attempts

        return severity

    def analyze_traffic_patterns(self, logs_df):
        """Analyze traffic patterns for anomalies"""
        # Requests per IP
        ip_counts = logs_df['ip_address'].value_counts()

        # Identify potential DDoS sources (high request volume)
        threshold = ip_counts.quantile(0.95)  # Top 5% of requesters
        ddos_candidates = ip_counts[ip_counts > threshold]

        # Time-based analysis
        logs_df['hour'] = logs_df['timestamp'].dt.hour
        hourly_traffic = logs_df.groupby('hour').size()

        # Status code analysis
        status_analysis = logs_df['status_code'].value_counts()

        return {
            'top_requesters': ip_counts.head(10),
            'ddos_candidates': ddos_candidates,
            'hourly_traffic': hourly_traffic,
            'status_codes': status_analysis,
            'total_requests': len(logs_df),
            'unique_ips': logs_df['ip_address'].nunique()
        }

    def generate_security_report(self, logs_df, attacks_df, traffic_analysis):
        """Generate comprehensive security report"""
        report = {
            'summary': {
                'total_requests': len(logs_df),
                'attacks_detected': len(attacks_df),
                'suspicious_ips': len(self.suspicious_ips),
                'attack_success_rate': len(attacks_df[attacks_df['severity'] >= 8]) / max(len(attacks_df), 1) * 100
            },
            'top_attacks': attacks_df['attack_type'].value_counts() if not attacks_df.empty else {},
            'high_severity_attacks': attacks_df[attacks_df['severity'] >= 8] if not attacks_df.empty else pd.DataFrame(),
            'traffic_analysis': traffic_analysis
        }

        return report

    def create_visualizations(self, logs_df, attacks_df, traffic_analysis):
        """Create security visualization dashboard"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Network Security Analysis Dashboard', fontsize=16, fontweight='bold')

        # 1. Attack Types Distribution
        if not attacks_df.empty:
            attack_counts = attacks_df['attack_type'].value_counts()
            axes[0, 0].pie(attack_counts.values, labels=attack_counts.index, autopct='%1.1f%%')
            axes[0, 0].set_title('Attack Types Distribution')
        else:
            axes[0, 0].text(0.5, 0.5, 'No attacks detected', ha='center', va='center')
            axes[0, 0].set_title('Attack Types Distribution')

        # 2. Hourly Traffic Pattern
        traffic_analysis['hourly_traffic'].plot(kind='bar', ax=axes[0, 1], color='skyblue')
        axes[0, 1].set_title('Traffic by Hour')
        axes[0, 1].set_xlabel('Hour of Day')
        axes[0, 1].set_ylabel('Number of Requests')
        axes[0, 1].tick_params(axis='x', rotation=45)

        # 3. Status Code Distribution
        status_counts = traffic_analysis['status_codes']
        colors = ['green' if x == 200 else 'orange' if x < 400 else 'red' for x in status_counts.index]
        axes[0, 2].bar(range(len(status_counts)), status_counts.values, color=colors)
        axes[0, 2].set_title('HTTP Status Codes')
        axes[0, 2].set_xlabel('Status Code')
        axes[0, 2].set_ylabel('Count')
        axes[0, 2].set_xticks(range(len(status_counts)))
        axes[0, 2].set_xticklabels(status_counts.index, rotation=45)

        # 4. Top Requesting IPs
        top_ips = traffic_analysis['top_requesters'].head(10)
        axes[1, 0].barh(range(len(top_ips)), top_ips.values)
        axes[1, 0].set_title('Top 10 Requesting IPs')
        axes[1, 0].set_xlabel('Number of Requests')
        axes[1, 0].set_yticks(range(len(top_ips)))
        axes[1, 0].set_yticklabels([f"{ip[:15]}..." if len(ip) > 15 else ip for ip in top_ips.index])

        # 5. Attack Severity Distribution
        if not attacks_df.empty:
            severity_counts = attacks_df['severity'].value_counts().sort_index()
            axes[1, 1].bar(severity_counts.index, severity_counts.values,
                          color=['green' if x < 5 else 'orange' if x < 8 else 'red' for x in severity_counts.index])
            axes[1, 1].set_title('Attack Severity Distribution')
            axes[1, 1].set_xlabel('Severity Level')
            axes[1, 1].set_ylabel('Number of Attacks')
        else:
            axes[1, 1].text(0.5, 0.5, 'No attacks detected', ha='center', va='center')
            axes[1, 1].set_title('Attack Severity Distribution')

        # 6. Timeline of Attacks
        if not attacks_df.empty:
            attacks_timeline = attacks_df.set_index('timestamp').resample('H').size()
            attacks_timeline.plot(ax=axes[1, 2], color='red', linewidth=2)
            axes[1, 2].set_title('Attack Timeline')
            axes[1, 2].set_xlabel('Time')
            axes[1, 2].set_ylabel('Attacks per Hour')
            axes[1, 2].tick_params(axis='x', rotation=45)
        else:
            axes[1, 2].text(0.5, 0.5, 'No attacks to plot', ha='center', va='center')
            axes[1, 2].set_title('Attack Timeline')

        plt.tight_layout()
        plt.show()

    def password_strength_checker(self, password):
        """Check password strength and provide recommendations"""
        score = 0
        feedback = []

        # Length check
        if len(password) >= 12:
            score += 2
        elif len(password) >= 8:
            score += 1
        else:
            feedback.append("Password should be at least 8 characters long")

        # Character variety checks
        if re.search(r'[a-z]', password):
            score += 1
        else:
            feedback.append("Add lowercase letters")

        if re.search(r'[A-Z]', password):
            score += 1
        else:
            feedback.append("Add uppercase letters")

        if re.search(r'\d', password):
            score += 1
        else:
            feedback.append("Add numbers")

        if re.search(r'[!@#$%^&*(),.?":{}|<>]', password):
            score += 1
        else:
            feedback.append("Add special characters")

        # Common patterns check
        common_patterns = ['123', 'abc', 'qwerty', 'password', 'admin']
        if any(pattern in password.lower() for pattern in common_patterns):
            score -= 1
            feedback.append("Avoid common patterns and dictionary words")

        strength_levels = {
            0: "Very Weak",
            1: "Very Weak",
            2: "Weak",
            3: "Fair",
            4: "Good",
            5: "Strong",
            6: "Very Strong"
        }

        return {
            'score': max(0, score),
            'strength': strength_levels[min(6, max(0, score))],
            'feedback': feedback
        }

    def hash_password(self, password, salt=None):
        """Demonstrate secure password hashing"""
        if salt is None:
            salt = base64.b64encode(np.random.bytes(32)).decode()

        # Using SHA-256 with salt (in practice, use bcrypt or scrypt)
        salted_password = password + salt
        hashed = hashlib.sha256(salted_password.encode()).hexdigest()

        return {
            'hash': hashed,
            'salt': salt,
            'algorithm': 'SHA-256'
        }

def main():
    print("🔒 Cybersecurity Network Security Analyzer")
    print("=" * 50)

    # Initialize analyzer
    analyzer = NetworkSecurityAnalyzer()

    # Generate sample data
    print("📊 Generating sample network logs...")
    logs_df = analyzer.generate_sample_logs(1500)
    print(f"Generated {len(logs_df)} log entries")

    # Detect attacks
    print("\n🔍 Analyzing logs for security threats...")
    attacks_df = analyzer.detect_attacks(logs_df)
    print(f"Detected {len(attacks_df)} potential attacks")

    # Analyze traffic patterns
    print("\n📈 Analyzing traffic patterns...")
    traffic_analysis = analyzer.analyze_traffic_patterns(logs_df)

    # Generate security report
    print("\n📋 Generating security report...")
    security_report = analyzer.generate_security_report(logs_df, attacks_df, traffic_analysis)

    # Display summary
    print("\n" + "="*50)
    print("SECURITY ANALYSIS SUMMARY")
    print("="*50)
    print(f"Total Requests Analyzed: {security_report['summary']['total_requests']:,}")
    print(f"Attacks Detected: {security_report['summary']['attacks_detected']}")
    print(f"Suspicious IP Addresses: {security_report['summary']['suspicious_ips']}")
    print(f"Attack Success Rate: {security_report['summary']['attack_success_rate']:.1f}%")

    if not attacks_df.empty:
        print(f"\nTop Attack Types:")
        for attack_type, count in security_report['top_attacks'].head().items():
            print(f"  • {attack_type.replace('_', ' ').title()}: {count}")

        print(f"\nHigh Severity Attacks (8+): {len(security_report['high_severity_attacks'])}")

    print(f"\nTraffic Statistics:")
    print(f"  • Unique IP Addresses: {traffic_analysis['unique_ips']:,}")
    print(f"  • Peak Hour Traffic: {traffic_analysis['hourly_traffic'].max()} requests")

    # Create visualizations
    print("\n📊 Creating security dashboard...")
    analyzer.create_visualizations(logs_df, attacks_df, traffic_analysis)

    # Password security demonstration
    print("\n🔐 Password Security Analysis")
    print("-" * 30)
    test_passwords = ["123456", "password", "MyP@ssw0rd!", "Tr0ub4dor&3", "correcthorsebatterystaple"]

    for pwd in test_passwords:
        result = analyzer.password_strength_checker(pwd)
        print(f"Password: {'*' * len(pwd)} | Strength: {result['strength']} (Score: {result['score']}/6)")
        if result['feedback']:
            print(f"  Recommendations: {', '.join(result['feedback'])}")

    # Demonstrate password hashing
    print(f"\n🔐 Secure Password Hashing Example:")
    sample_password = "MySecurePassword123!"
    hash_result = analyzer.hash_password(sample_password)
    print(f"Original: {sample_password}")
    print(f"Hash: {hash_result['hash'][:32]}...")
    print(f"Salt: {hash_result['salt'][:16]}...")
    print(f"Algorithm: {hash_result['algorithm']}")

    # Display sample attack data
    if not attacks_df.empty:
        print(f"\n⚠️  Recent High-Severity Attacks:")
        high_severity = attacks_df[attacks_df['severity'] >= 8].head()
        for idx, attack in high_severity.iterrows():
            print(f"  • {attack['attack_type'].replace('_', ' ').title()} from {attack['ip_address']} "
                  f"(Severity: {attack['severity']}/10)")

    print(f"\n✅ Analysis complete! Check the visualizations above for detailed insights.")

if __name__ == "__main__":
    main()